# Protein Distribution

This notebook explores the distribution of protein in the DisProt dataset, taking into 
account for their release date, existing PDB structures and whether they have been used 
for CAID evaluation.

In [1]:
# Setup the environment
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/.julia/dev/IDRBindingDataset.jl/scripts`


In [2]:
# Load the necessary packages.
# Important functions for the analysis are defined in the IDRBindingDataset package, rather
# than in this notebook, so that they can be easily reused in other contexts.
using CodecZlib
using DataFrames
using IDRBindingDataset
using JSON
using MIToS

## Load the DisProt dataset

We will first load the DisProt dataset (in JSON format), and for each protein—identified by
its UniProt accession number—we will retrieve the release version tag (`YYYY_MM`) of the
oldest annotated intrinsic disorder region (IDR). For that, we will first create a
DataFrame of IDRs, and then we will group the IDRs by their corresponding UniProt
accession number, and for each group we will retrieve the oldest region release tag.
This will lead to two `const` DataFrames we can later use throughout the notebook:
`IDRS` and `PROTEINS`.

### DisProt JSON format (API search output)

Source: DisProt API docs (`https://disprot.org/api`) and API search output (`https://disprot.org/api/search?...`).

According to the DisProt API docs for `api/search`, the response contains two top-level keys:
- `size`: number of protein entries returned
- `data`: array of protein entries

Each protein entry includes identifiers (`disprot_id`, `acc`), sequence-level metadata,
a protein-level `released` tag (`YYYY_MM`), and a `regions` array.

Each item in `regions` is one annotation record and includes fields such as:
- `region_id`, `start`, `end`
- region-level `released` (`YYYY_MM`)
- ontology labels (`term_id`, `term_name`, `term_namespace`)
- evidence/publication metadata (`ec_id`, `reference_id`, etc.)

Important: two region records can share the same `start` and `end` when they annotate
different concepts (for example structural state vs structural transition) on the same segment.

In [3]:
# Define shared raw-data root and source DisProt file.
const RAW = joinpath(@__DIR__, "..", "data", "raw")

const DISPROT_JSON_PATH = joinpath(RAW, "disprot", "disprot_regions.json.gz")

DISPROT_JSON_PATH

"/home/diego.zea/.julia/dev/IDRBindingDataset.jl/scripts/../data/raw/disprot/disprot_regions.json.gz"

In [4]:
# Load the compressed DisProt JSON payload and keep the protein-level entries.
const DISPROT_PAYLOAD = open(DISPROT_JSON_PATH) do io
    JSON.parse(read(GzipDecompressorStream(io), String))
end

const DISPROT_ENTRIES = DISPROT_PAYLOAD["data"]

println("Loaded $(length(DISPROT_ENTRIES)) proteins from $(DISPROT_JSON_PATH)")

Loaded 3199 proteins from /home/diego.zea/.julia/dev/IDRBindingDataset.jl/scripts/../data/raw/disprot/disprot_regions.json.gz


In [5]:
# Flatten the nested regions into an IDR-level DataFrame.
const IDRS = let rows = NamedTuple[]
    for protein in DISPROT_ENTRIES
        acc = protein["acc"]
        disprot_id = protein["disprot_id"]
        protein_released = protein["released"]

        for region in protein["regions"]
            push!(
                rows,
                (
                    acc = acc,
                    disprot_id = disprot_id,
                    protein_released = protein_released,
                    region_id = region["region_id"],
                    term_id = region["term_id"],
                    term_name = region["term_name"],
                    term_namespace = region["term_namespace"],
                    start = Int(region["start"]),
                    stop = Int(region["end"]),
                    released = region["released"],
                ),
            )
        end
    end
    DataFrame(rows)
end

first(IDRS, 10)

Row,acc,disprot_id,protein_released,region_id,term_id,term_name,term_namespace,start,stop,released
,String,String,String,String,String,String,String,Int64,Int64,String
1,P03265,DP00003,2016_10,DP00003r002,IDPO:0000002,disorder,Structural state,294,334,2022_03
2,P03265,DP00003,2016_10,DP00003r004,IDPO:0000002,disorder,Structural state,454,464,2022_03
3,P49913,DP00004,2016_10,DP00004r001,IDPO:0000002,disorder,Structural state,134,170,2022_03
4,P49913,DP00004,2016_10,DP00004r002,IDPO:0000011,disorder to order,Structural transition,134,170,2022_03
5,P49913,DP00004,2016_10,DP00004r004,GO:0019835,cytolysis,Biological process,134,170,2022_06
6,P49913,DP00004,2016_10,DP00004r005,GO:1990000,amyloid fibril formation,Biological process,150,162,2022_12
7,P49913,DP00004,2016_10,DP00004r006,GO:1990000,amyloid fibril formation,Biological process,150,162,2022_12
8,P03045,DP00005,2016_10,DP00005r001,IDPO:0000002,disorder,Structural state,1,107,2022_03
9,P03045,DP00005,2016_10,DP00005r004,IDPO:0000002,disorder,Structural state,1,107,2022_03


In [6]:
# Group IDRs by UniProt accession and retain release information per protein.
const PROTEINS = let grouped = groupby(IDRS, :acc)
    combine(
        grouped,
        :disprot_id => first => :disprot_id,
        :protein_released => first => :protein_released,
        :released => (x -> minimum(parse_disprot_release.(x))) => :oldest_idr_release,
        :released =>
            (x -> maximum(parse_disprot_release.(x))) => :most_recent_idr_release,
        nrow => :n_idrs,
    )
end

sort!(PROTEINS, :oldest_idr_release)
first(PROTEINS, 10)

Row,acc,disprot_id,protein_released,oldest_idr_release,most_recent_idr_release,n_idrs
,String,String,String,Date,Date,Int64
1,P03265,DP00003,2016_10,2022-03-01,2022-03-01,2
2,P49913,DP00004,2016_10,2022-03-01,2022-12-01,5
3,P03045,DP00005,2016_10,2022-03-01,2022-03-01,16
4,P00004,DP00006,2016_10,2022-03-01,2022-12-01,4
5,P27695,DP00007,2016_10,2022-03-01,2022-03-01,4
6,Q64693,DP00008,2016_10,2022-03-01,2022-03-01,5
7,P13569,DP00012,2016_10,2022-03-01,2024-06-01,18
8,P0DN86,DP00013,2016_10,2022-03-01,2022-03-01,4
9,P61926,DP00015,2016_10,2022-03-01,2022-03-01,19


In [7]:
# Print a compact summary so we can quickly sanity-check the generated tables.
println("Total IDRs: $(nrow(IDRS))")
println("Total proteins: $(nrow(PROTEINS))")
println(
    "Oldest protein entry release among proteins: $(minimum(PROTEINS.protein_released))",
)
println(
    "Most recent protein entry release among proteins: $(maximum(PROTEINS.protein_released))",
)
println("Oldest IDR release among proteins: $(minimum(PROTEINS.oldest_idr_release))")
println(
    "Most recent IDR release among proteins: $(maximum(PROTEINS.most_recent_idr_release))",
)

Total IDRs: 13396
Total proteins: 3199
Oldest protein entry release among proteins: 2016_10
Most recent protein entry release among proteins: 2025_06
Oldest IDR release among proteins: 2022-03-01
Most recent IDR release among proteins: 2025-12-01


### UniProt to PDB Mapping from DisProt Cross References

This section builds a UniProt-to-PDB mapping using region-level DisProt cross
references. The final table has one row per unique `(acc, pdb)` pair.

In [8]:
# Extract region-level PDB cross references using shared package logic.
const REGION_LEVEL_PDB_REFS = collect_disprot_region_level_pdb_refs(DISPROT_ENTRIES)

println(
    "Region-level UniProt-PDB rows (cleaned IDs, repeated pairs possible): $(nrow(REGION_LEVEL_PDB_REFS))",
)
first(REGION_LEVEL_PDB_REFS, 10)

Region-level UniProt-PDB rows (cleaned IDs, repeated pairs possible): 6749


Row,acc,pdb,pdb_reference
,String,String,String
1,A0A024B7W1,6LD1,6LD1
2,A0A024B7W1,6LD1,6LD1
3,A0A024B7W1,7BSD,7BSD
4,A0A090N8E9,4MI0,4MI0
5,A0A097F4I8,7ELA,7ELA
6,A0A097F4I8,7ELA,7ELA
7,A0A0B4JDC9,6Q9M,6Q9M
8,A0A0B4JDC9,6Q9M,6Q9M
9,A0A0C4DGY3,6XT9,6XT9


In [9]:
# pdb is the cleaned canonical PDB ID used for joins/lookups
# pdb_reference is the original PDB ID as listed in DisProt kept for provenance/debugging
REGION_LEVEL_PDB_REFS[REGION_LEVEL_PDB_REFS.pdb .!= REGION_LEVEL_PDB_REFS.pdb_reference, :]

Row,acc,pdb,pdb_reference
,String,String,String
1,A0A3Q8IT45,5OSG,5OSG
2,E6Y5X0,2MM8,2mm8
3,G3V7P1,2NPS,2nps
4,G3V7P1,2NPS,2nps
5,O14936,1ZL8,1zl8
6,O70480,2NPS,2nps
7,O70480,2NPS,2nps
8,O88597,3BL2,3bl2
9,P01555,1S5B,1S5B:A


In [10]:
# Collapse repeated region references into one row per UniProt-PDB pair.
const UNIPROT_PDB_MAP = build_unique_uniprot_pdb_map(REGION_LEVEL_PDB_REFS)

println("Unique UniProt-PDB pairs: $(nrow(UNIPROT_PDB_MAP))")
first(UNIPROT_PDB_MAP, 10)

Unique UniProt-PDB pairs: 3792


Row,acc,pdb
,String,String
1,A0A024B7W1,6LD1
2,A0A024B7W1,7BSD
3,A0A090N8E9,4MI0
4,A0A097F4I8,7ELA
5,A0A0B4JDC9,6Q9M
6,A0A0C4DGY3,6XT9
7,A0A0G2KTI4,6NPJ
8,A0A0G2KTI4,6NPL
9,A0A0H2UWN8,7N10


In [11]:
# Show UniProt-PDB pairs statistics.
println("Total region-level PDB references: $(nrow(REGION_LEVEL_PDB_REFS))")
println("Unique UniProt-PDB pairs: $(nrow(UNIPROT_PDB_MAP))")
n_repeated_rows = nrow(REGION_LEVEL_PDB_REFS) - nrow(UNIPROT_PDB_MAP)
println("Repeated region rows collapsed: $(n_repeated_rows)")
println("Proteins with at least one PDB mapping: $(length(unique(UNIPROT_PDB_MAP.acc)))")

Total region-level PDB references: 6749
Unique UniProt-PDB pairs: 3792
Repeated region rows collapsed: 2957
Proteins with at least one PDB mapping: 2221


## Load SIFTS tables

We load chain-level SIFTS summary tables with `MIToS.SIFTS.read_file`.
These files map PDB chains to UniProt and CATH.

In [12]:
# Define paths to SIFTS summary tables.
const SIFTS_UNIPROT_PATH = joinpath(RAW, "sifts", "pdb_chain_uniprot.csv.gz")

const SIFTS_CATH_UNIPROT_PATH = joinpath(RAW, "sifts", "pdb_chain_cath_uniprot.csv.gz")

# Load the SIFTS summary tables using MIToS.
const SIFTS_UNIPROT = MIToS.SIFTS.read_file(SIFTS_UNIPROT_PATH, MIToS.SIFTS.SIFTSCSV)

const SIFTS_CATH_UNIPROT =
    MIToS.SIFTS.read_file(SIFTS_CATH_UNIPROT_PATH, MIToS.SIFTS.SIFTSCSV)

println("SIFTS UniProt table size: $(size(SIFTS_UNIPROT.table))")
println("SIFTS CATH+UniProt table size: $(size(SIFTS_CATH_UNIPROT.table))")

SIFTS UniProt table size: (962516, 9)
SIFTS CATH+UniProt table size: (449473, 4)


In [13]:
# Inspect column names parsed by MIToS.
println("UniProt columns: $(SIFTS_UNIPROT.colnames)")
println("CATH+UniProt columns: $(SIFTS_CATH_UNIPROT.colnames)")

UniProt columns: [:PDB, :CHAIN, :SP_PRIMARY, :RES_BEG, :RES_END, :PDB_BEG, :PDB_END, :SP_BEG, :SP_END]
CATH+UniProt columns: [:PDB, :CHAIN, :SP_PRIMARY, :CATH_ID]


In [14]:
# Convert and preview the UniProt summary table.
const SIFTS_UNIPROT_DF = DataFrame(SIFTS_UNIPROT.table, SIFTS_UNIPROT.colnames)

first(SIFTS_UNIPROT_DF, 10)

Row,PDB,CHAIN,SP_PRIMARY,RES_BEG,RES_END,PDB_BEG,PDB_END,SP_BEG,SP_END
,String,String,String,String,String,String,String,String,String
1,101m,A,P02185,1,154,0,153,1,154
2,102l,A,P00720,1,40,1,40,1,40
3,102l,A,P00720,42,165,41,None,41,164
4,102m,A,P02185,1,154,0,153,1,154
5,103l,A,P00720,1,40,1,None,1,40
6,103l,A,P00720,44,167,41,None,41,164
7,103m,A,P02185,1,154,0,153,1,154
8,104l,A,P00720,1,44,1,44,1,44
9,104l,A,P00720,47,166,45,None,45,164


In [15]:
# Convert and preview the CATH+UniProt summary table.
const SIFTS_CATH_UNIPROT_DF =
    DataFrame(SIFTS_CATH_UNIPROT.table, SIFTS_CATH_UNIPROT.colnames)

first(SIFTS_CATH_UNIPROT_DF, 10)

Row,PDB,CHAIN,SP_PRIMARY,CATH_ID
,String,String,String,String
1,101m,A,P02185,101mA00
2,102l,A,P00720,102lA00
3,102m,A,P02185,102mA00
4,103l,A,P00720,103lA00
5,103m,A,P02185,103mA00
6,104l,A,P00720,104lA00
7,104l,B,P00720,104lB00
8,104m,A,P02185,104mA00
9,105m,A,P02185,105mA00


## Compare DisProt and SIFTS PDB cross mapping

We first take all unique DisProt UniProt accessions, map them to PDB using
`SIFTS_UNIPROT_DF`, and then perform an outer join against `UNIPROT_PDB_MAP`
to identify shared pairs and pairs found only on one side.

In [16]:
# Collect the unique UniProt IDs present in DisProt.
const DISPROT_UNIPROT_IDS = sort(unique(PROTEINS.acc))
const DISPROT_UNIPROT_ID_SET = Set(DISPROT_UNIPROT_IDS)

println("Unique DisProt UniProt IDs: $(length(DISPROT_UNIPROT_IDS))")
first(DISPROT_UNIPROT_IDS, 10)

Unique DisProt UniProt IDs: 3199


10-element Vector{String}:
 "A0A024B7W1"
 "A0A068MVV3"
 "A0A078IB52"
 "A0A090N8E9"
 "A0A097F4I8"
 "A0A0B4JDC9"
 "A0A0C4DGY3"
 "A0A0G2JXC5"
 "A0A0G2KTI4"
 "A0A0H2UWN8"

In [17]:
# Build UniProt-PDB pairs from SIFTS, restricted to DisProt UniProt IDs.
const SIFTS_UNIPROT_PDB_MAP =
    build_sifts_uniprot_pdb_map_for_disprot(SIFTS_UNIPROT_DF, DISPROT_UNIPROT_ID_SET)

println(
    "Unique SIFTS UniProt-PDB pairs for DisProt proteins: $(nrow(SIFTS_UNIPROT_PDB_MAP))",
)
first(SIFTS_UNIPROT_PDB_MAP, 10)

Unique SIFTS UniProt-PDB pairs for DisProt proteins: 51416


Row,acc,pdb
,String,String
1,A0A024B7W1,5GOZ
2,A0A024B7W1,5GP1
3,A0A024B7W1,5H30
4,A0A024B7W1,5H32
5,A0A024B7W1,5H37
6,A0A024B7W1,5IRE
7,A0A024B7W1,5IZ7
8,A0A024B7W1,5JMT
9,A0A024B7W1,5KQR


In [18]:
# Compare DisProt and SIFTS UniProt-PDB maps.
const DISPROT_SIFTS_PDB_COMPARISON =
    build_disprot_sifts_pdb_comparison(UNIPROT_PDB_MAP, SIFTS_UNIPROT_PDB_MAP)

compare_summary =
    combine(groupby(DISPROT_SIFTS_PDB_COMPARISON, :mapping_status), nrow => :n_pairs)
sort!(compare_summary, :mapping_status)
compare_summary

Row,mapping_status,n_pairs
,String,Int64
1,both,3680
2,only_disprot,112
3,only_sifts,47736


In [19]:
# Preview pairs that appear only in one source.
only_disprot =
    subset(DISPROT_SIFTS_PDB_COMPARISON, :mapping_status => ByRow(==("only_disprot")))

only_sifts =
    subset(DISPROT_SIFTS_PDB_COMPARISON, :mapping_status => ByRow(==("only_sifts")))

println("Pairs only in DisProt: $(nrow(only_disprot))")
println("Pairs only in SIFTS: $(nrow(only_sifts))")
first(only_disprot, 10), first(only_sifts, 10)

Pairs only in DisProt: 112
Pairs only in SIFTS: 47736


(10×3 DataFrame
 Row │ acc         pdb     mapping_status 
     │ String      String  String         
─────┼────────────────────────────────────
   1 │ A0A090N8E9  4MI0    only_disprot
   2 │ A0A0H3LBR3  4W4K    only_disprot
   3 │ A0A1L8HGL2  7FIL    only_disprot
   4 │ A0A2Z5UJ33  2IQH    only_disprot
   5 │ A0A5F9ZHA1  4D8O    only_disprot
   6 │ A0A5F9ZHA1  4RLV    only_disprot
   7 │ A0A5F9ZHA1  5Y4D    only_disprot
   8 │ A0A5F9ZHA1  5Y4F    only_disprot
   9 │ A0A5F9ZHA1  6M3Q    only_disprot
  10 │ A0A7M7G9Q4  2LID    only_disprot, 10×3 DataFrame
 Row │ acc         pdb     mapping_status 
     │ String      String  String         
─────┼────────────────────────────────────
   1 │ A0A024B7W1  5GOZ    only_sifts
   2 │ A0A024B7W1  5GP1    only_sifts
   3 │ A0A024B7W1  5H30    only_sifts
   4 │ A0A024B7W1  5H32    only_sifts
   5 │ A0A024B7W1  5H37    only_sifts
   6 │ A0A024B7W1  5IRE    only_sifts
   7 │ A0A024B7W1  5IZ7    only_sifts
   8 │ A0A024B7W1  5JMT    only_sifts
   9 │

#### Note on the discrepancies between DisProt and SIFTS mappings

- `4MI0` marked as `only_disprot` is associated with `A0A090N8E9` in DisProt, but SIFTS/PDB 
maps that PDB to `Q15910`.
- `2LID` marked as `only_disprot` is 35 amino acid residues long, and is not associated with 
any UniProt accession in SIFTS/PDB.

It is common that short sequences on PDB are not mapped to UniProt through SIFTS, but that 
has been described in the literature and annotated on DisProt. It can also be cases in 
which the UniProt accession number associated with a PDB structure has been updated, and 
the old accession number is still used in DisProt cross references. In any case, due to 
these discrepancies, we will merge the annotations list to assign DisProt proteins to 
PDB structures, and not rely on SIFTS mappings alone.

## DisProt and PDB mapping

Following the note above, we keep the merged DisProt/SIFTS mapping and retain
only `acc` and `pdb`.

In [20]:
# Build cleaned merged UniProt-PDB map plus release-date/status annotations.
const PDB_RELEASE_ANALYSIS =
    build_pdb_release_map_from_comparison(DISPROT_SIFTS_PDB_COMPARISON; batch_size = 300)

const DISPROT_PDB_MAP = PDB_RELEASE_ANALYSIS.disprot_pdb_map
# For how many DisProt PDB references did the regex-based PDB code extraction fail 
# to find a valid canonical 4-character PDB code?
const INVALID_MERGED_PDB_REFERENCES = PDB_RELEASE_ANALYSIS.invalid_merged_pdb_references

println(
    "Merged UniProt-PDB raw references: $(nrow(PDB_RELEASE_ANALYSIS.merged_pdb_references))",
)
println("Merged UniProt-PDB pairs after regex cleaning: $(nrow(DISPROT_PDB_MAP))")
println("Merged references with no valid PDB code: $(nrow(INVALID_MERGED_PDB_REFERENCES))")
first(DISPROT_PDB_MAP, 10)

Merged UniProt-PDB raw references: 51528
Merged UniProt-PDB pairs after regex cleaning: 51528
Merged references with no valid PDB code: 0


Row,acc,pdb
,String,String
1,A0A024B7W1,5GOZ
2,A0A024B7W1,5GP1
3,A0A024B7W1,5H30
4,A0A024B7W1,5H32
5,A0A024B7W1,5H37
6,A0A024B7W1,5IRE
7,A0A024B7W1,5IZ7
8,A0A024B7W1,5JMT
9,A0A024B7W1,5KQR


### PDB release dates and failing-ID analysis

This section summarizes release-date lookup outcomes and highlights failing UniProt-PDB 
mappings for manual review. Run `scripts/08_pdb_release_dates.jl` to persist the table
of PDB release dates and failing mappings as a CSV in `data/derived/pdb/`.

In [21]:
# Analyze release-date lookup failures without writing files from the notebook.
const DISPROT_PDB_MAP_WITH_RELEASE = PDB_RELEASE_ANALYSIS.disprot_pdb_map_with_release
const FAILED_PDB_LOOKUPS = PDB_RELEASE_ANALYSIS.failed_pdb_lookups

println("Rows in cleaned merged map: $(nrow(DISPROT_PDB_MAP_WITH_RELEASE))")
println("Rows with missing PDB release date: $(nrow(FAILED_PDB_LOOKUPS))")
println(
    "Unique PDB codes with missing release date: $(length(unique(FAILED_PDB_LOOKUPS.pdb)))",
)

first(DISPROT_PDB_MAP_WITH_RELEASE, 10)

Rows in cleaned merged map: 51530
Rows with missing PDB release date: 10
Unique PDB codes with missing release date: 9


Row,acc,pdb,pdb_release_date,pdb_lookup_classification,rcsb_http_status,rcsb_status,rcsb_status_code,replacement_pdb
,String,String,Date?,String,Int64,String?,String?,String?
1,A0A024B7W1,5GOZ,2016-12-07,current,0,missing,missing,missing
2,A0A024B7W1,5GP1,2016-12-07,current,0,missing,missing,missing
3,A0A024B7W1,5H30,2016-11-30,current,0,missing,missing,missing
4,A0A024B7W1,5H32,2016-11-30,current,0,missing,missing,missing
5,A0A024B7W1,5H37,2016-11-30,current,0,missing,missing,missing
6,A0A024B7W1,5IRE,2016-03-30,current,0,missing,missing,missing
7,A0A024B7W1,5IZ7,2016-05-25,current,0,missing,missing,missing
8,A0A024B7W1,5JMT,2016-05-25,current,0,missing,missing,missing
9,A0A024B7W1,5KQR,2016-09-14,current,0,missing,missing,missing


In [22]:
INVALID_MERGED_PDB_REFERENCES

Row,acc,pdb_reference,failure_reason
,String,String,String


In [23]:
select(
    FAILED_PDB_LOOKUPS,
    :acc,
    :pdb,
    :pdb_lookup_classification,
    :rcsb_status,
    :rcsb_status_code,
    :replacement_pdb,
)

Row,acc,pdb,pdb_lookup_classification,rcsb_status,rcsb_status_code,replacement_pdb
,String,String,String,String?,String?,String?
1,Q9S446,12TW,invalid_or_not_found,missing,missing,missing
2,Q9RY65,1KC9,obsolete,REMOVED,OBS,missing
3,P07140,1QON,obsolete,REMOVED,OBS,6XYU
4,P62326,1UY5,obsolete,REMOVED,TRSF,missing
5,E6PBU9,2XZM,obsolete,REMOVED,OBS,4V5O
6,I7MK25,2XZM,obsolete,REMOVED,OBS,4V5O
7,P47047,3L9O,obsolete,REMOVED,OBS,4QU4
8,P01023,4ACQ,obsolete,REMOVED,OBS,6TAV
9,Q7T2G3,60SW,invalid_or_not_found,missing,missing,missing


In [24]:
# Are replacement PDB codes present in the DISPROT_PDB_MAP_WITH_RELEASE table?
const REPLACEMENT_PDB_CODES =
    Set{String}(filter(!ismissing, FAILED_PDB_LOOKUPS.replacement_pdb))

Set{String} with 5 elements:
  "6TAV"
  "6XYU"
  "6QZP"
  "4QU4"
  "4V5O"

In [26]:
DISPROT_PDB_MAP_WITH_RELEASE[
    [pdb in REPLACEMENT_PDB_CODES for pdb in DISPROT_PDB_MAP_WITH_RELEASE.pdb],
    :pdb,
] |> Set

Set{String} with 5 elements:
  "6TAV"
  "6XYU"
  "6QZP"
  "4QU4"
  "4V5O"